# PySpark ETL
This notebook focuses on performing ETL (Extract, Transform, Load) operations on the data ingested from the FPL API. We will use PySpark to clean, transform, and prepare the data for model training and prediction.

## 1. Include required libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * # Import all functions
from pyspark.sql.window import Window
import os
from datetime import datetime, date

## 2. Initialize Spark Session
We initialize a Spark session, which is the entry point to any Spark functionality.

In [2]:
spark = SparkSession.builder.appName("gameweek-prophet-etl").getOrCreate()

your 131072x1 screen size is bogus. expect trouble
25/03/30 10:49:29 WARN Utils: Your hostname, LAPTOP-3B4JAVBH resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/03/30 10:49:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/30 10:49:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 3. Read Data
We read the CSV files generated by earlier steps into Spark dataframes:
* fpl_historical/XXXX_XX_merged_gw.csv
* fpl_historical/XXXX_XX_fixtures.csv
* fpl_historical/XXXX_XX_teams.csv
* fpl_historical/XXXX_XX_players_raw.csv
* fpl_api/2024_25_player_history.csv
* fpl_api/2024_25_fixtures.csv
* fpl_api/2024_25_teams.csv
* fpl_api/2024_25_elements.csv

We add the following metadata to the dataframes: 
* season_code
* source_filename
* source_date

In [3]:
import os
import re
from pyspark.sql.functions import lit

def add_season_code(df, filepath):
    """
    Adds a 'season' column to the DataFrame based on the filepath.

    Args:
        df: The PySpark DataFrame.
        filepath: The filepath of the CSV file.

    Returns:
        The DataFrame with the added 'season' column.
    """

    # Extract the season from the filepath using a regular expression
    # Assuming filepaths have a pattern like 'season_2022_2023.csv'

    filename = os.path.basename(filepath)
    match = re.search(r'.*(\d{4}_\d{2})_.*\.csv', filename)
    if match:
        season_code = match.group(1)  # e.g., "2022_23"
    else:
        raise ValueError(f"No season pattern found in filepath: {filepath}")

    # Add the 'season' column to the DataFrame
    df = df.withColumn("season_code", lit(season_code))
    return df

def add_source_metadata(df, filepath):
    """
    Adds source metadata columns to the DataFrame.

    Args:
        df: The PySpark DataFrame.
        filepath: The filepath of the CSV file.

    Returns:
        The DataFrame with the added source metadata columns.
    """


    # Extract the directory containing the datetime string
    datetime_dir = os.path.dirname(filepath)  # Gets the directory path
    datetime_string = os.path.basename(datetime_dir)  # Gets the last directory name
    filename = os.path.basename(filepath)  # Gets the filename

    # Check if the directory name matches the datetime pattern
    if re.match(r'\d{8}_\d{6}', datetime_string):
        # Convert the datetime string to a datetime object
        source_datetime = datetime.strptime(datetime_string, "%Y%m%d_%H%M%S")
    
    else:
        raise ValueError(f"No source datetime found in filepath: {filepath}")

    # Add the source filepath and datetime columns
    df = df.withColumn("source_filename", lit(filename))\
            .withColumn("source_datetime", lit(source_datetime))
    return df

### 3.1 Historical Gameweek Data

In [4]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType, TimestampType

# Define the schema for the FPL historical merged player gameweek data
schema_merged_gw_2022_2023 = StructType([
    StructField("name", StringType(), True),
    StructField("position", StringType(), True),
    StructField("team", StringType(), True),
    StructField("xP", FloatType(), True),
    StructField("assists", IntegerType(), True),
    StructField("bonus", IntegerType(), True),
    StructField("bps", IntegerType(), True),
    StructField("clean_sheets", IntegerType(), True),
    StructField("creativity", FloatType(), True),
    StructField("element", IntegerType(), True),
    StructField("expected_assists", FloatType(), True),
    StructField("expected_goal_involvements", FloatType(), True),
    StructField("expected_goals", FloatType(), True),
    StructField("expected_goals_conceded", FloatType(), True),
    StructField("fixture", IntegerType(), True),
    StructField("goals_conceded", IntegerType(), True),
    StructField("goals_scored", IntegerType(), True),
    StructField("ict_index", FloatType(), True),
    StructField("influence", FloatType(), True),
    StructField("kickoff_time", TimestampType(), True),
    StructField("minutes", IntegerType(), True),
    StructField("opponent_team", IntegerType(), True),
    StructField("own_goals", IntegerType(), True),
    StructField("penalties_missed", IntegerType(), True),
    StructField("penalties_saved", IntegerType(), True),
    StructField("red_cards", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("saves", IntegerType(), True),
    StructField("selected", IntegerType(), True),
    StructField("starts", IntegerType(), True),
    StructField("team_a_score", IntegerType(), True),
    StructField("team_h_score", IntegerType(), True),
    StructField("threat", FloatType(), True),
    StructField("total_points", IntegerType(), True),
    StructField("transfers_balance", IntegerType(), True),
    StructField("transfers_in", IntegerType(), True),
    StructField("transfers_out", IntegerType(), True),
    StructField("value", FloatType(), True),
    StructField("was_home", BooleanType(), True),
    StructField("yellow_cards", IntegerType(), True),
    StructField("GW", IntegerType(), True)
])

schema_merged_gw_2024 = StructType([
    StructField("name", StringType(), True),
    StructField("position", StringType(), True),
    StructField("team", StringType(), True),
    StructField("xP", FloatType(), True),
    StructField("assists", IntegerType(), True),
    StructField("bonus", IntegerType(), True),
    StructField("bps", IntegerType(), True),
    StructField("clean_sheets", IntegerType(), True),
    StructField("creativity", FloatType(), True),
    StructField("element", IntegerType(), True),
    StructField("expected_assists", FloatType(), True),
    StructField("expected_goal_involvements", FloatType(), True),
    StructField("expected_goals", FloatType(), True),
    StructField("expected_goals_conceded", FloatType(), True),
    StructField("fixture", IntegerType(), True),
    StructField("goals_conceded", IntegerType(), True),
    StructField("goals_scored", IntegerType(), True),
    StructField("ict_index", FloatType(), True),
    StructField("influence", FloatType(), True),
    StructField("kickoff_time", TimestampType(), True),
    StructField("minutes", IntegerType(), True),
    StructField("modified", BooleanType(), True),  # Added for 2024_25_merged_gw.csv
    StructField("opponent_team", IntegerType(), True),
    StructField("own_goals", IntegerType(), True),
    StructField("penalties_missed", IntegerType(), True),
    StructField("penalties_saved", IntegerType(), True),
    StructField("red_cards", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("saves", IntegerType(), True),
    StructField("selected", IntegerType(), True),
    StructField("starts", IntegerType(), True),
    StructField("team_a_score", IntegerType(), True),
    StructField("team_h_score", IntegerType(), True),
    StructField("threat", FloatType(), True),
    StructField("total_points", IntegerType(), True),
    StructField("transfers_balance", IntegerType(), True),
    StructField("transfers_in", IntegerType(), True),
    StructField("transfers_out", IntegerType(), True),
    StructField("value", FloatType(), True),
    StructField("was_home", BooleanType(), True),
    StructField("yellow_cards", IntegerType(), True),
    StructField("GW", IntegerType(), True)
])

data_dir = "../data/raw/fpl_historical/20250313_225438/"
input_files_schema = {
    # os.path.join(data_dir, "2024_25_merged_gw_fixed.csv"): schema_merged_gw_2024,
    os.path.join(data_dir, "2023_24_merged_gw.csv"): schema_merged_gw_2022_2023,
    os.path.join(data_dir, "2022_23_merged_gw.csv"): schema_merged_gw_2022_2023,
}

dfs = []

# Read each file with the schema
for filepath, schema in input_files_schema.items():
    # Read the CSV file into a DataFrame
    input_df = spark.read.csv(filepath, header=True, schema=schema)

    # Add the season code
    df_with_season = add_season_code(input_df, filepath)

    # Add source filename and datetime
    df_with_metadata = add_source_metadata(df_with_season, filepath)

    # Append the DataFrame to the list
    dfs.append(df_with_metadata)


# Combine all DataFrames into one
historical_merged_gw_df = dfs[0]
for df in dfs:
    historical_merged_gw_df = historical_merged_gw_df.unionByName(df, allowMissingColumns=True)

# Show the combined DataFrame
historical_merged_gw_df.show()

25/03/30 10:49:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+--------+-------------+----+-------+-----+---+------------+----------+-------+----------------+--------------------------+--------------+-----------------------+-------+--------------+------------+---------+---------+-------------------+-------+-------------+---------+----------------+---------------+---------+-----+-----+--------+------+------------+------------+------+------------+-----------------+------------+-------------+-----+--------+------------+---+-----------+--------------------+-------------------+
|                name|position|         team|  xP|assists|bonus|bps|clean_sheets|creativity|element|expected_assists|expected_goal_involvements|expected_goals|expected_goals_conceded|fixture|goals_conceded|goals_scored|ict_index|influence|       kickoff_time|minutes|opponent_team|own_goals|penalties_missed|penalties_saved|red_cards|round|saves|selected|starts|team_a_score|team_h_score|threat|total_points|transfers_balance|transfers_in|transfers_out|value|wa

### 3.2 Historical Fixture Data

In [5]:
from pyspark.sql.types import StructType, StructField, IntegerType, BooleanType, StringType, TimestampType

# Define the schema for the FPL historical fixture data
fixture_schema = StructType([
    StructField("code", IntegerType(), True),
    StructField("event", IntegerType(), True),
    StructField("finished", BooleanType(), True),
    StructField("finished_provisional", BooleanType(), True),
    StructField("id", IntegerType(), True),
    StructField("kickoff_time", TimestampType(), True),
    StructField("minutes", IntegerType(), True),
    StructField("provisional_start_time", BooleanType(), True),
    StructField("started", BooleanType(), True),
    StructField("team_a", IntegerType(), True),
    StructField("team_a_score", IntegerType(), True),
    StructField("team_h", IntegerType(), True),
    StructField("team_h_score", IntegerType(), True),
    StructField("stats", StringType(), True),  # Likely JSON or dict stored as string
    StructField("team_h_difficulty", IntegerType(), True),
    StructField("team_a_difficulty", IntegerType(), True),
    StructField("pulse_id", IntegerType(), True),
])

data_dir = "../data/raw/fpl_historical/fixtures/20250322_155111"
input_files_schema = {
    os.path.join(data_dir, "2022_23_fixtures.csv"): fixture_schema,
    os.path.join(data_dir, "2023_24_fixtures.csv"): fixture_schema,
}

dfs = []

# Read each file with the schema
for filepath, schema in input_files_schema.items():
    # Read the CSV file into a DataFrame
    input_df = spark.read.csv(filepath, header=True, schema=schema, nullValue=None)

    # Add the season code
    df_with_season = add_season_code(input_df, filepath)

    # Add source filename and datetime
    df_with_metadata = add_source_metadata(df_with_season, filepath)

    # Append the DataFrame to the list
    dfs.append(df_with_metadata)


# Combine all DataFrames into one
historical_fixtures_df = dfs[0]
for df in dfs:
    historical_fixtures_df = historical_fixtures_df.unionByName(df, allowMissingColumns=True)

# Show the combined DataFrame
historical_fixtures_df.show()

+-------+-----+--------+--------------------+---+-------------------+-------+----------------------+-------+------+------------+------+------------+--------------------+-----------------+-----------------+--------+-----------+--------------------+-------------------+
|   code|event|finished|finished_provisional| id|       kickoff_time|minutes|provisional_start_time|started|team_a|team_a_score|team_h|team_h_score|               stats|team_h_difficulty|team_a_difficulty|pulse_id|season_code|     source_filename|    source_datetime|
+-------+-----+--------+--------------------+---+-------------------+-------+----------------------+-------+------+------------+------+------------+--------------------+-----------------+-----------------+--------+-----------+--------------------+-------------------+
|2292810|    1|    true|                true|  1|2022-08-06 05:00:00|     90|                 false|   true|     1|           2|     7|           0|[{'identifier': '...|                4|         

### 3.3 Historical Team Data

In [6]:
from pyspark.sql.types import StructType, StructField, IntegerType, BooleanType, StringType

# Define the schema for the FPL historical teams data
teams_schema = StructType([
    StructField("code", IntegerType(), True),
    StructField("draw", IntegerType(), True),
    StructField("form", StringType(), True),
    StructField("id", IntegerType(), True),
    StructField("loss", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("played", IntegerType(), True),
    StructField("points", IntegerType(), True),
    StructField("position", IntegerType(), True),
    StructField("short_name", StringType(), True),
    StructField("strength", IntegerType(), True),
    StructField("team_division", IntegerType(), True),
    StructField("unavailable", BooleanType(), True),
    StructField("win", IntegerType(), True),
    StructField("strength_overall_home", IntegerType(), True),
    StructField("strength_overall_away", IntegerType(), True),
    StructField("strength_attack_home", IntegerType(), True),
    StructField("strength_attack_away", IntegerType(), True),
    StructField("strength_defence_home", IntegerType(), True),
    StructField("strength_defence_away", IntegerType(), True),
    StructField("pulse_id", IntegerType(), True)
])

data_dir = "../data/raw/fpl_historical/teams/20250322_154945"
input_files_schema = {
    os.path.join(data_dir, "2022_23_teams.csv"): teams_schema,
    os.path.join(data_dir, "2023_24_teams.csv"): teams_schema,
}

dfs = []

# Read each file with the schema
for filepath, schema in input_files_schema.items():
    # Read the CSV file into a DataFrame
    input_df = spark.read.csv(filepath, header=True, schema=schema)

    # Add the season code
    df_with_season = add_season_code(input_df, filepath)

    # Add source filename and datetime
    df_with_metadata = add_source_metadata(df_with_season, filepath)

    # Append the DataFrame to the list
    dfs.append(df_with_metadata)


# Combine all DataFrames into one
historical_teams_df = dfs[0]
for df in dfs:
    historical_teams_df = historical_teams_df.unionByName(df, allowMissingColumns=True)

# Show the combined DataFrame
historical_teams_df.show()

+----+----+----+---+----+--------------+------+------+--------+----------+--------+-------------+-----------+---+---------------------+---------------------+--------------------+--------------------+---------------------+---------------------+--------+-----------+-----------------+-------------------+
|code|draw|form| id|loss|          name|played|points|position|short_name|strength|team_division|unavailable|win|strength_overall_home|strength_overall_away|strength_attack_home|strength_attack_away|strength_defence_home|strength_defence_away|pulse_id|season_code|  source_filename|    source_datetime|
+----+----+----+---+----+--------------+------+------+--------+----------+--------+-------------+-----------+---+---------------------+---------------------+--------------------+--------------------+---------------------+---------------------+--------+-----------+-----------------+-------------------+
|   3|   0|NULL|  1|   0|       Arsenal|     0|     0|       0|       ARS|       4|        

### 3.4 Historical Player Data

In [7]:
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, BooleanType, StringType

# Define the schema for the FPL historical players data
players_raw_schema = StructType([
    StructField('assists', IntegerType(), True), 
    StructField('bonus', IntegerType(), True), 
    StructField('bps', IntegerType(), True), 
    StructField('chance_of_playing_next_round', StringType(), True), 
    StructField('chance_of_playing_this_round', StringType(), True), 
    StructField('clean_sheets', IntegerType(), True), 
    StructField('clean_sheets_per_90', FloatType(), True), 
    StructField('code', IntegerType(), True), 
    StructField('corners_and_indirect_freekicks_order', StringType(), True), 
    StructField('corners_and_indirect_freekicks_text', StringType(), True), 
    StructField('cost_change_event', IntegerType(), True), 
    StructField('cost_change_event_fall', IntegerType(), True), 
    StructField('cost_change_start', IntegerType(), True), 
    StructField('cost_change_start_fall', IntegerType(), True), 
    StructField('creativity', FloatType(), True), 
    StructField('creativity_rank', IntegerType(), True), 
    StructField('creativity_rank_type', IntegerType(), True), 
    StructField('direct_freekicks_order', StringType(), True), 
    StructField('direct_freekicks_text', StringType(), True), 
    StructField('dreamteam_count', IntegerType(), True), 
    StructField('element_type', IntegerType(), True), 
    StructField('ep_next', StringType(), True), 
    StructField('ep_this', FloatType(), True), 
    StructField('event_points', IntegerType(), True), 
    StructField('expected_assists', FloatType(), True), 
    StructField('expected_assists_per_90', FloatType(), True), 
    StructField('expected_goal_involvements', FloatType(), True), 
    StructField('expected_goal_involvements_per_90', FloatType(), True), 
    StructField('expected_goals', FloatType(), True), 
    StructField('expected_goals_conceded', FloatType(), True), 
    StructField('expected_goals_conceded_per_90', FloatType(), True), 
    StructField('expected_goals_per_90', FloatType(), True), 
    StructField('first_name', StringType(), True), 
    StructField('form', FloatType(), True), 
    StructField('form_rank', IntegerType(), True), 
    StructField('form_rank_type', IntegerType(), True), 
    StructField('goals_conceded', IntegerType(), True), 
    StructField('goals_conceded_per_90', FloatType(), True), 
    StructField('goals_scored', IntegerType(), True), 
    StructField('ict_index', FloatType(), True), 
    StructField('ict_index_rank', IntegerType(), True), 
    StructField('ict_index_rank_type', IntegerType(), True), 
    StructField('id', IntegerType(), True), 
    StructField('in_dreamteam', BooleanType(), True), 
    StructField('influence', FloatType(), True), 
    StructField('influence_rank', IntegerType(), True), 
    StructField('influence_rank_type', IntegerType(), True), 
    StructField('minutes', IntegerType(), True), 
    StructField('news', StringType(), True), 
    StructField('news_added', StringType(), True), 
    StructField('now_cost', IntegerType(), True), 
    StructField('now_cost_rank', IntegerType(), True), 
    StructField('now_cost_rank_type', IntegerType(), True), 
    StructField('own_goals', IntegerType(), True), 
    StructField('penalties_missed', IntegerType(), True), 
    StructField('penalties_order', StringType(), True), 
    StructField('penalties_saved', IntegerType(), True), 
    StructField('penalties_text', StringType(), True), 
    StructField('photo', StringType(), True), 
    StructField('points_per_game', FloatType(), True), 
    StructField('points_per_game_rank', IntegerType(), True), 
    StructField('points_per_game_rank_type', IntegerType(), True), 
    StructField('red_cards', IntegerType(), True), 
    StructField('saves', IntegerType(), True), 
    StructField('saves_per_90', FloatType(), True), 
    StructField('second_name', StringType(), True), 
    StructField('selected_by_percent', FloatType(), True), 
    StructField('selected_rank', IntegerType(), True), 
    StructField('selected_rank_type', IntegerType(), True), 
    StructField('special', BooleanType(), True), 
    StructField('squad_number', StringType(), True), 
    StructField('starts', IntegerType(), True), 
    StructField('starts_per_90', FloatType(), True), 
    StructField('status', StringType(), True), 
    StructField('team', IntegerType(), True), 
    StructField('team_code', IntegerType(), True), 
    StructField('threat', FloatType(), True), 
    StructField('threat_rank', IntegerType(), True), 
    StructField('threat_rank_type', IntegerType(), True), 
    StructField('total_points', IntegerType(), True), 
    StructField('transfers_in', IntegerType(), True), 
    StructField('transfers_in_event', IntegerType(), True), 
    StructField('transfers_out', IntegerType(), True), 
    StructField('transfers_out_event', IntegerType(), True), 
    StructField('value_form', FloatType(), True), 
    StructField('value_season', FloatType(), True), 
    StructField('web_name', StringType(), True), 
    StructField('yellow_cards', IntegerType(), True)
])

data_dir = "../data/raw/fpl_historical/players_raw/20250324_062401"
input_files_schema = {
    os.path.join(data_dir, "2022_23_players_raw.csv"): players_raw_schema,
    os.path.join(data_dir, "2023_24_players_raw.csv"): players_raw_schema,
}

dfs = []

# Read each file with the schema
for filepath, schema in input_files_schema.items():
    # Read the CSV file into a DataFrame
    input_df = spark.read.csv(filepath, header=True, schema=schema)

    # Add the season code
    df_with_season = add_season_code(input_df, filepath)

    # Add source filename and datetime
    df_with_metadata = add_source_metadata(df_with_season, filepath)

    # Append the DataFrame to the list
    dfs.append(df_with_metadata)


# Combine all DataFrames into one
historical_players_df = dfs[0]
for df in dfs:
    historical_players_df = historical_players_df.unionByName(df, allowMissingColumns=True)

# Show the combined DataFrame
historical_players_df.show()

+-------+-----+---+----------------------------+----------------------------+------------+-------------------+------+------------------------------------+-----------------------------------+-----------------+----------------------+-----------------+----------------------+----------+---------------+--------------------+----------------------+---------------------+---------------+------------+-------+-------+------------+----------------+-----------------------+--------------------------+---------------------------------+--------------+-----------------------+------------------------------+---------------------+----------+----+---------+--------------+--------------+---------------------+------------+---------+--------------+-------------------+---+------------+---------+--------------+-------------------+-------+--------------------+--------------------+--------+-------------+------------------+---------+----------------+---------------+---------------+--------------+----------+--------

### 3.5 Current Gameweek Data

In [8]:
from pyspark.sql.types import StructType, StructField, FloatType, BooleanType, TimestampType, FloatType, StringType

player_history_schema = StructType([
    StructField("element", FloatType(), True),
    StructField("fixture", FloatType(), True),
    StructField("opponent_team", FloatType(), True),
    StructField("total_points", FloatType(), True),
    StructField("was_home", BooleanType(), True),
    StructField("kickoff_time", TimestampType(), True),
    StructField("team_h_score", FloatType(), True),
    StructField("team_a_score", FloatType(), True),
    StructField("round", FloatType(), True),
    StructField("modified", BooleanType(), True),
    StructField("minutes", FloatType(), True),
    StructField("goals_scored", FloatType(), True),
    StructField("assists", FloatType(), True),
    StructField("clean_sheets", FloatType(), True),
    StructField("goals_conceded", FloatType(), True),
    StructField("own_goals", FloatType(), True),
    StructField("penalties_saved", FloatType(), True),
    StructField("penalties_missed", FloatType(), True),
    StructField("yellow_cards", FloatType(), True),
    StructField("red_cards", FloatType(), True),
    StructField("saves", FloatType(), True),
    StructField("bonus", FloatType(), True),
    StructField("bps", FloatType(), True),
    StructField("influence", FloatType(), True),
    StructField("creativity", FloatType(), True),
    StructField("threat", FloatType(), True),
    StructField("ict_index", FloatType(), True),
    StructField("starts", FloatType(), True),
    StructField("expected_goals", FloatType(), True),
    StructField("expected_assists", FloatType(), True),
    StructField("expected_goal_involvements", FloatType(), True),
    StructField("expected_goals_conceded", FloatType(), True),
    StructField("mng_win", FloatType(), True),
    StructField("mng_draw", FloatType(), True),
    StructField("mng_loss", FloatType(), True),
    StructField("mng_underdog_win", FloatType(), True),
    StructField("mng_underdog_draw", FloatType(), True),
    StructField("mng_clean_sheets", FloatType(), True),
    StructField("mng_goals_scored", FloatType(), True),
    StructField("value", FloatType(), True),
    StructField("transfers_balance", FloatType(), True),
    StructField("selected", FloatType(), True),
    StructField("transfers_in", FloatType(), True),
    StructField("transfers_out", FloatType(), True)
])

colums_to_cast = {
    "element":"int",
    "fixture":"int",
    "opponent_team":"int",
    "total_points":"int",
    "team_h_score":"int",
    "team_a_score":"int",
    "round":"int",
    "minutes":"int",
    "goals_scored":"int",
    "assists":"int",
    "clean_sheets":"int",
    "goals_conceded":"int",
    "own_goals":"int",
    "penalties_saved":"int",
    "penalties_missed":"int",
    "yellow_cards":"int",
    "red_cards":"int",
    "saves":"int",
    "bonus":"int",
    "bps":"int",
    "starts":"int",
    "mng_win":"int",
    "mng_draw":"int",
    "mng_loss":"int",
    "mng_underdog_win":"int",
    "mng_underdog_draw":"int",
    "mng_clean_sheets":"int",
    "mng_goals_scored":"int",
    "value":"int",
    "transfers_balance":"int",
    "selected":"int",
    "transfers_in":"int",
    "transfers_out":"int",
}

data_dir = "../data/raw/fpl_api/20250323_104114/"
filepath = os.path.join(data_dir, "2024_25_player_history.csv")

# Read each file with the schema
# Read the CSV file into a DataFrame
input_df = spark.read.csv(filepath, header=True, schema=player_history_schema, nullValue=None)

# Cast the team scores to integers and fill null values with None
cleaned_df = input_df
for col_name, cast_type in colums_to_cast.items():
    cleaned_df = cleaned_df.withColumn(col_name, when(col(col_name).cast(cast_type).isNull(), None).otherwise(col(col_name).cast(cast_type)))

# Add the season code
df_with_season = add_season_code(cleaned_df, filepath)

# Add source filename and datetime
df_with_metadata = add_source_metadata(df_with_season, filepath)

# Show the player history DataFrame
api_player_history_df = df_with_metadata
api_player_history_df.show()

+-------+-------+-------------+------------+--------+-------------------+------------+------------+-----+--------+-------+------------+-------+------------+--------------+---------+---------------+----------------+------------+---------+-----+-----+---+---------+----------+------+---------+------+--------------+----------------+--------------------------+-----------------------+-------+--------+--------+----------------+-----------------+----------------+----------------+-----+-----------------+--------+------------+-------------+-----------+--------------------+-------------------+
|element|fixture|opponent_team|total_points|was_home|       kickoff_time|team_h_score|team_a_score|round|modified|minutes|goals_scored|assists|clean_sheets|goals_conceded|own_goals|penalties_saved|penalties_missed|yellow_cards|red_cards|saves|bonus|bps|influence|creativity|threat|ict_index|starts|expected_goals|expected_assists|expected_goal_involvements|expected_goals_conceded|mng_win|mng_draw|mng_loss|mng

### 3.6 Current Fixture Data

In [9]:
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, BooleanType, StringType, TimestampType

# Define the schema for the FPL API fixture data
fixture_schema = StructType([
    StructField("code", IntegerType(), True),
    StructField("event", IntegerType(), True),
    StructField("finished", BooleanType(), True),
    StructField("finished_provisional", BooleanType(), True),
    StructField("id", IntegerType(), True),
    StructField("kickoff_time", TimestampType(), True),
    StructField("minutes", IntegerType(), True),
    StructField("provisional_start_time", BooleanType(), True),
    StructField("started", BooleanType(), True),
    StructField("team_a", IntegerType(), True),
    StructField("team_a_score", FloatType(), True),
    StructField("team_h", IntegerType(), True),
    StructField("team_h_score", FloatType(), True),
    StructField("stats", StringType(), True),  # Likely JSON or dict stored as string
    StructField("team_h_difficulty", IntegerType(), True),
    StructField("team_a_difficulty", IntegerType(), True),
    StructField("pulse_id", IntegerType(), True),
    StructField("team_id", IntegerType(), True),
])

data_dir = "../data/raw/fpl_api/20250323_003842"
filepath = os.path.join(data_dir, "2024_25_fixtures.csv")

# Read the CSV file into a DataFrame
input_df = spark.read.csv(filepath, header=True, schema=fixture_schema, nullValue=None)

# Cast the team scores to integers and fill null values with None
cleaned_df = input_df.withColumn("team_a_score", when(col("team_a_score").cast("int").isNull(), None).otherwise(col("team_a_score").cast("int"))) \
                     .withColumn("team_h_score", when(col("team_h_score").cast("int").isNull(), None).otherwise(col("team_h_score").cast("int")))

# Add the season code
df_with_season = add_season_code(cleaned_df, filepath)

# Add source filename and datetime
df_with_metadata = add_source_metadata(df_with_season, filepath)

# Combine into fixtures dataframe
api_fixtures_df = df_with_metadata

# Show the combined DataFrame
api_fixtures_df.show()

+-------+-----+--------+--------------------+---+-------------------+-------+----------------------+-------+------+------------+------+------------+--------------------+-----------------+-----------------+--------+-------+-----------+--------------------+-------------------+
|   code|event|finished|finished_provisional| id|       kickoff_time|minutes|provisional_start_time|started|team_a|team_a_score|team_h|team_h_score|               stats|team_h_difficulty|team_a_difficulty|pulse_id|team_id|season_code|     source_filename|    source_datetime|
+-------+-----+--------+--------------------+---+-------------------+-------+----------------------+-------+------+------------+------+------------+--------------------+-----------------+-----------------+--------+-------+-----------+--------------------+-------------------+
|2444471|    1|    true|                true|  2|2024-08-18 00:00:00|     90|                 false|   true|    20|           0|     1|           2|[{'identifier': '...|   

### 3.7 Current Team Data

In [10]:
from pyspark.sql.types import StructType, StructField, IntegerType, BooleanType, StringType, FloatType, DateType, TimestampType

# Define the schema for the FPL API teams data
teams_schema = StructType([
    StructField('code', IntegerType(), True), 
    StructField('draw', IntegerType(), True), 
    StructField('form', StringType(), True), 
    StructField('id', IntegerType(), True), 
    StructField('loss', IntegerType(), True), 
    StructField('name', StringType(), True), 
    StructField('played', IntegerType(), True), 
    StructField('points', IntegerType(), True), 
    StructField('position', IntegerType(), True), 
    StructField('short_name', StringType(), True), 
    StructField('strength', IntegerType(), True), 
    StructField('team_division', StringType(), True), 
    StructField('unavailable', BooleanType(), True), 
    StructField('win', IntegerType(), True), 
    StructField('strength_overall_home', IntegerType(), True), 
    StructField('strength_overall_away', IntegerType(), True), 
    StructField('strength_attack_home', IntegerType(), True), 
    StructField('strength_attack_away', IntegerType(), True), 
    StructField('strength_defence_home', IntegerType(), True), 
    StructField('strength_defence_away', IntegerType(), True), 
    StructField('pulse_id', IntegerType(), True)
])

data_dir = "../data/raw/fpl_api/20250323_104114"
filepath = os.path.join(data_dir, "2024_25_teams.csv")

# Read each file with the schema
# Read the CSV file into a DataFrame
input_df = spark.read.csv(filepath, header=True, schema=teams_schema)

# Add the season code
df_with_season = add_season_code(input_df, filepath)

# Add source filename and datetime
df_with_metadata = add_source_metadata(df_with_season, filepath)

# Show the teams DataFrame
api_teams_df = df_with_metadata
api_teams_df.show()

+----+----+----+---+----+--------------+------+------+--------+----------+--------+-------------+-----------+---+---------------------+---------------------+--------------------+--------------------+---------------------+---------------------+--------+-----------+-----------------+-------------------+
|code|draw|form| id|loss|          name|played|points|position|short_name|strength|team_division|unavailable|win|strength_overall_home|strength_overall_away|strength_attack_home|strength_attack_away|strength_defence_home|strength_defence_away|pulse_id|season_code|  source_filename|    source_datetime|
+----+----+----+---+----+--------------+------+------+--------+----------+--------+-------------+-----------+---+---------------------+---------------------+--------------------+--------------------+---------------------+---------------------+--------+-----------+-----------------+-------------------+
|   3|   0|NULL|  1|   0|       Arsenal|     0|     0|       2|       ARS|       4|        

### 3.8 Current Player Data

In [11]:
from pyspark.sql.types import StructType, StructField, IntegerType, BooleanType, StringType, FloatType, DateType, TimestampType

# Define the schema for the FPL API elements data
elements_schema = StructType([
    StructField('can_transact', BooleanType(), True), 
    StructField('can_select', BooleanType(), True), 
    StructField('chance_of_playing_next_round', FloatType(), True), 
    StructField('chance_of_playing_this_round', FloatType(), True), 
    StructField('code', IntegerType(), True), 
    StructField('cost_change_event', IntegerType(), True), 
    StructField('cost_change_event_fall', IntegerType(), True), 
    StructField('cost_change_start', IntegerType(), True), 
    StructField('cost_change_start_fall', IntegerType(), True), 
    StructField('dreamteam_count', IntegerType(), True), 
    StructField('element_type', IntegerType(), True), 
    StructField('ep_next', FloatType(), True), 
    StructField('ep_this', FloatType(), True), 
    StructField('event_points', IntegerType(), True), 
    StructField('first_name', StringType(), True), 
    StructField('form', FloatType(), True), 
    StructField('id', IntegerType(), True), 
    StructField('in_dreamteam', BooleanType(), True), 
    StructField('news', StringType(), True), 
    StructField('news_added', TimestampType(), True), 
    StructField('now_cost', IntegerType(), True), 
    StructField('photo', StringType(), True), 
    StructField('points_per_game', FloatType(), True), 
    StructField('removed', BooleanType(), True), 
    StructField('second_name', StringType(), True), 
    StructField('selected_by_percent', FloatType(), True), 
    StructField('special', BooleanType(), True), 
    StructField('squad_number', StringType(), True), 
    StructField('status', StringType(), True), 
    StructField('team', IntegerType(), True), 
    StructField('team_code', IntegerType(), True), 
    StructField('total_points', IntegerType(), True), 
    StructField('transfers_in', IntegerType(), True), 
    StructField('transfers_in_event', IntegerType(), True), 
    StructField('transfers_out', IntegerType(), True), 
    StructField('transfers_out_event', IntegerType(), True), 
    StructField('value_form', FloatType(), True), 
    StructField('value_season', FloatType(), True), 
    StructField('web_name', StringType(), True), 
    StructField('region', FloatType(), True), 
    StructField('team_join_date', DateType(), True), 
    StructField('birth_date', DateType(), True), 
    StructField('has_temporary_code', BooleanType(), True), 
    StructField('opta_code', StringType(), True), 
    StructField('minutes', IntegerType(), True), 
    StructField('goals_scored', IntegerType(), True), 
    StructField('assists', IntegerType(), True), 
    StructField('clean_sheets', IntegerType(), True), 
    StructField('goals_conceded', IntegerType(), True), 
    StructField('own_goals', IntegerType(), True), 
    StructField('penalties_saved', IntegerType(), True), 
    StructField('penalties_missed', IntegerType(), True), 
    StructField('yellow_cards', IntegerType(), True), 
    StructField('red_cards', IntegerType(), True), 
    StructField('saves', IntegerType(), True), 
    StructField('bonus', IntegerType(), True), 
    StructField('bps', IntegerType(), True), 
    StructField('influence', FloatType(), True), 
    StructField('creativity', FloatType(), True), 
    StructField('threat', FloatType(), True), 
    StructField('ict_index', FloatType(), True), 
    StructField('starts', IntegerType(), True), 
    StructField('expected_goals', FloatType(), True), 
    StructField('expected_assists', FloatType(), True), 
    StructField('expected_goal_involvements', FloatType(), True), 
    StructField('expected_goals_conceded', FloatType(), True), 
    StructField('mng_win', IntegerType(), True), 
    StructField('mng_draw', IntegerType(), True), 
    StructField('mng_loss', IntegerType(), True), 
    StructField('mng_underdog_win', IntegerType(), True), 
    StructField('mng_underdog_draw', IntegerType(), True), 
    StructField('mng_clean_sheets', IntegerType(), True), 
    StructField('mng_goals_scored', IntegerType(), True), 
    StructField('influence_rank', IntegerType(), True), 
    StructField('influence_rank_type', IntegerType(), True), 
    StructField('creativity_rank', IntegerType(), True), 
    StructField('creativity_rank_type', IntegerType(), True), 
    StructField('threat_rank', IntegerType(), True), 
    StructField('threat_rank_type', IntegerType(), True), 
    StructField('ict_index_rank', IntegerType(), True), 
    StructField('ict_index_rank_type', IntegerType(), True), 
    StructField('corners_and_indirect_freekicks_order', FloatType(), True), 
    StructField('corners_and_indirect_freekicks_text', StringType(), True), 
    StructField('direct_freekicks_order', FloatType(), True), 
    StructField('direct_freekicks_text', StringType(), True), 
    StructField('penalties_order', FloatType(), True), 
    StructField('penalties_text', StringType(), True), 
    StructField('expected_goals_per_90', FloatType(), True), 
    StructField('saves_per_90', FloatType(), True), 
    StructField('expected_assists_per_90', FloatType(), True), 
    StructField('expected_goal_involvements_per_90', FloatType(), True), 
    StructField('expected_goals_conceded_per_90', FloatType(), True), 
    StructField('goals_conceded_per_90', FloatType(), True), 
    StructField('now_cost_rank', IntegerType(), True), 
    StructField('now_cost_rank_type', IntegerType(), True), 
    StructField('form_rank', IntegerType(), True), 
    StructField('form_rank_type', IntegerType(), True), 
    StructField('points_per_game_rank', IntegerType(), True), 
    StructField('points_per_game_rank_type', IntegerType(), True), 
    StructField('selected_rank', IntegerType(), True), 
    StructField('selected_rank_type', IntegerType(), True), 
    StructField('starts_per_90', FloatType(), True), 
    StructField('clean_sheets_per_90', FloatType(), True)
])

data_dir = "../data/raw/fpl_api/20250323_104114"
filepath = os.path.join(data_dir, "2024_25_elements.csv")

# Read each file with the schema
# Read the CSV file into a DataFrame
input_df = spark.read.csv(filepath, header=True, schema=elements_schema)

# Add the season code
df_with_season = add_season_code(input_df, filepath)

# Add source filename and datetime
df_with_metadata = add_source_metadata(df_with_season, filepath)

# Show the elements DataFrame
api_elements_df = df_with_metadata
api_elements_df.show()

+------------+----------+----------------------------+----------------------------+------+-----------------+----------------------+-----------------+----------------------+---------------+------------+-------+-------+------------+----------+----+---+------------+--------------------+--------------------+--------+----------+---------------+-------+--------------------+-------------------+-------+------------+------+----+---------+------------+------------+------------------+-------------+-------------------+----------+------------+------------+------+--------------+----------+------------------+---------+-------+------------+-------+------------+--------------+---------+---------------+----------------+------------+---------+-----+-----+---+---------+----------+------+---------+------+--------------+----------------+--------------------------+-----------------------+-------+--------+--------+----------------+-----------------+----------------+----------------+--------------+------------

## 4. Transform & Clean Data
We'll transform the data by merging input datasets and handle data cleaning tasks such as dealing with missing values. But first let's pick which features data points we are interested in so we don't waste effort on cleaning data that is not required.

### 4.1 Transform & Clean Player Data
In this section we'll transform and clean player data.

**Player Dimensions**
* Player name
* Player code
* Team name
* Team code
* Opponent team name
* Opponent team code
* Position
* Kickoff time
* Gameweek
* Fixture code
* Season code

**Player Metrics**
* Goals scored
* Goals conceded
* Assists
* Expected assists
* Expected goal involvements
* Expected goals
* Expected goals conceded
* Penalties scored
* Penalties missed
* Penalties saved
* Saves
* BPS (Bonus Points System)
* Points
* Minutes
* Yellow cards
* Red cards
* Own goals
* Starts
* Was home
* Influence
* Creativity
* Threat
* ICT index
* Clean sheets

### 4.1.1 Merge Historical Player Data
In this section we merge historical gameweek data with historical player data and historical team data.

In [12]:
# Rename join keys in historical_merged_gw_df *before* the join
historical_player_data = historical_merged_gw_df\
    .withColumnRenamed("element", "player_id")\
    .withColumnRenamed("team", "team_name")\
    .withColumnRenamed("opponent_team", "opponent_team_id")\
    .withColumnRenamed("fixture", "fixture_id")\

# Rename join keys in historical_players_df *before* the join
historical_players_unique = historical_players_df\
    .select("id", "code", "first_name", "second_name", "season_code")\
    .withColumnRenamed("id", "player_id")\
    .withColumnRenamed("code", "player_code")\
    .distinct()

# Rename join keys in historical_teams_df *before* the join
historical_teams_unique = historical_teams_df\
    .select("id", "code", "name", "season_code")\
    .withColumnRenamed("id", "team_id")\
    .withColumnRenamed("code", "team_code")\
    .withColumnRenamed("name", "team_name")\
    .distinct()

# Rename join keys in historical_teams_df *before* the join
historical_opponent_teams_unique = historical_teams_df\
    .select("id", "code", "name", "season_code")\
    .withColumnRenamed("id", "opponent_team_id")\
    .withColumnRenamed("code", "opponent_team_code")\
    .withColumnRenamed("name", "opponent_team_name")\
    .distinct()

# Rename join keys in historical_teams_df *before* the join
historical_opponent_teams_unique = historical_teams_df\
    .select("id", "code", "name", "season_code")\
    .withColumnRenamed("id", "opponent_team_id")\
    .withColumnRenamed("code", "opponent_team_code")\
    .withColumnRenamed("name", "opponent_team_name")\
    .distinct()

# Rename join keys in historical_fixtures_df *before* the join
historical_fixtures_unique = historical_fixtures_df\
    .select("id", "code", "season_code")\
    .withColumnRenamed("id", "fixture_id")\
    .withColumnRenamed("code", "fixture_code")\
    .distinct()

# Join historical_merged_gw_df with historical_players_unique and historical_teams_unique
historical_player_data = historical_player_data\
    .join(
        historical_players_unique,
        (["player_id", "season_code"]),
        "inner",
    )\
    .join(
        historical_teams_unique,
        (["team_name", "season_code"]),
        "inner"
    )\
    .join(
        historical_opponent_teams_unique,
        (["opponent_team_id", "season_code"]),
        "inner"
    )\
    .join(
        historical_fixtures_unique,
        (["fixture_id", "season_code"]),
        "inner"
    )

print(f"Number of rows in historical_merged_gw_df: {historical_merged_gw_df.count()}")
print(f"Number of rows in historical_players_unique: {historical_players_unique.count()}")
print(f"Number of rows in historical_teams_unique: {historical_teams_unique.count()}")
print(f"Number of rows in historical_fixtures_unique: {historical_fixtures_unique.count()}")
print(f"Number of rows in historical_player_data: {historical_player_data.count()}")

# Show the merged dataframe
historical_player_data.show()

Number of rows in historical_merged_gw_df: 85955
Number of rows in historical_players_unique: 1643
Number of rows in historical_teams_unique: 40
Number of rows in historical_fixtures_unique: 760
Number of rows in historical_player_data: 85955
+----------+-----------+----------------+-------------+---------+--------------------+--------+----+-------+-----+---+------------+----------+----------------+--------------------------+--------------+-----------------------+--------------+------------+---------+---------+-------------------+-------+---------+----------------+---------------+---------+-----+-----+--------+------+------------+------------+------+------------+-----------------+------------+-------------+-----+--------+------------+---+--------------------+-------------------+-----------+----------+-----------------+-------+---------+------------------+------------------+------------+
|fixture_id|season_code|opponent_team_id|    team_name|player_id|                name|position|  xP|

### 4.1.2 Clean Historical Player Data
In this section we clean historical player data.

In [13]:
player_data_columns = [
    "player_name",
    "player_code",
    "team_name",
    "team_code",
    "opponent_team_name",
    "opponent_team_code",
    "position",
    "kickoff_time",
    "gameweek",
    "round",
    "fixture_code",
    "season_code",
    "goals_scored",
    "goals_conceded",
    "assists",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals",
    "expected_goals_conceded",
    "penalties_missed",
    "penalties_saved",
    "saves",
    "bps",
    "minutes",
    "yellow_cards",
    "red_cards",
    "own_goals",
    "starts",
    "was_home",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "clean_sheets",
    "total_points",
]

# Rename columns in historical_player_data
cleaned_historical_player_data = historical_player_data\
    .withColumnRenamed("name", "player_name")\
    .withColumnRenamed("GW", "gameweek")

# Select required columns
cleaned_historical_player_data = cleaned_historical_player_data.select(player_data_columns)

# Show the selected columns
cleaned_historical_player_data.show()

+--------------------+-----------+-------------+---------+------------------+------------------+--------+-------------------+--------+-----+------------+-----------+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+------------+
|         player_name|player_code|    team_name|team_code|opponent_team_name|opponent_team_code|position|       kickoff_time|gameweek|round|fixture_code|season_code|goals_scored|goals_conceded|assists|expected_assists|expected_goal_involvements|expected_goals|expected_goals_conceded|penalties_missed|penalties_saved|saves|bps|minutes|yellow_cards|red_cards|own_goals|starts|was_home|influence|creativity|threat|ict_index|clean_sheets|total_points|
+--------------------+-----------+-------------+---------+------------------+------------------+------

### 4.1.3 Merge Current Player Data
In this section we merge historical gameweek data with historical player data and historical team data.

In [14]:
# Rename join keys in api_player_history_df *before* the join
current_player_data = api_player_history_df\
    .withColumnRenamed("element", "player_id")\
    .withColumnRenamed("fixture", "fixture_id")\
    .withColumnRenamed("opponent_team", "opponent_team_id")\

# Rename join keys in api_fixtures_df *before* the join
current_fixtures_unique = api_fixtures_df\
    .select("id", "code", "team_a", "team_h", "event", "season_code")\
    .withColumnRenamed("id", "fixture_id")\
    .withColumnRenamed("code", "fixture_code")\
    .withColumnRenamed("team_a", "away_team_id")\
    .withColumnRenamed("team_h", "home_team_id")\
    .withColumnRenamed("event", "gameweek")\
    .distinct()

# Rename join keys in api_elements_df *before* the join
current_players_unique = api_elements_df\
    .select("id", "code", "element_type", "first_name", "second_name", "season_code")\
    .withColumnRenamed("id", "player_id")\
    .withColumnRenamed("code", "player_code")\
    .distinct()

# Rename join keys in api_teams_df *before* the join
current_teams_unique = api_teams_df\
    .select("id", "code", "name", "season_code")\
    .withColumnRenamed("id", "team_id")\
    .withColumnRenamed("code", "team_code")\
    .withColumnRenamed("name", "team_name")\
    .distinct()

# Rename join keys in api_teams_df *before* the join
current_opponent_teams_unique = api_teams_df\
    .select("id", "code", "name", "season_code")\
    .withColumnRenamed("id", "opponent_team_id")\
    .withColumnRenamed("code", "opponent_team_code")\
    .withColumnRenamed("name", "opponent_team_name")\
    .distinct()

# Join current_player_data with current_fixtures_unique
# Calculate team id from fixture
current_player_data = current_player_data\
    .join(
        current_players_unique,
        (["player_id", "season_code"]),
        "inner"
    )\
    .join(
        current_fixtures_unique,
        (["fixture_id", "season_code"]),
        "inner"
    )\
    .withColumn(
        "team_id", 
        when(
            col("was_home") == True,
            col("home_team_id")
        ).otherwise(col("away_team_id"))
    )\
    .join(
        current_teams_unique,
        (["team_id", "season_code"]),
        "inner"
    )\
    .join(
        current_opponent_teams_unique,
        (["opponent_team_id", "season_code"]),
        "inner"
    )

print(f"Number of rows in api_player_history_df: {api_player_history_df.count()}")
print(f"Number of rows in api_fixtures_df: {api_fixtures_df.count()}")
print(f"Number of rows in api_elements_df: {api_elements_df.count()}")
print(f"Number of rows in api_teams_df: {api_teams_df.count()}")
print(f"Number of rows in current_player_data: {current_player_data.count()}")

# Show the merged dataframe
current_player_data.show()

Number of rows in api_player_history_df: 20347
Number of rows in api_fixtures_df: 760
Number of rows in api_elements_df: 789
Number of rows in api_teams_df: 20
Number of rows in current_player_data: 20347
+----------------+-----------+-------+----------+---------+------------+--------+-------------------+------------+------------+-----+--------+-------+------------+-------+------------+--------------+---------+---------------+----------------+------------+---------+-----+-----+---+---------+----------+------+---------+------+--------------+----------------+--------------------------+-----------------------+-------+--------+--------+----------------+-----------------+----------------+----------------+-----+-----------------+--------+------------+-------------+--------------------+-------------------+-----------+------------+----------+---------------+------------+------------+------------+--------+---------+---------+------------------+------------------+
|opponent_team_id|season_code|t

### 4.1.4 Clean Current Player Data
In this section we clean current player data.

In [15]:
# Rename columns in current_player_data
cleaned_current_player_data = current_player_data\
    .withColumn(
        "player_name",
        concat(col("first_name"), lit(" "), col("second_name")))\
    .withColumn(
        "position",
        when(col("element_type") == 1, "GK")\
        .when(col("element_type") == 2, "DEF")\
        .when(col("element_type") == 3, "MID")\
        .when(col("element_type") == 4, "FWD")\
    )

# Select required columns
cleaned_current_player_data = cleaned_current_player_data.select(player_data_columns)

# Show the selected columns
cleaned_current_player_data.show()

+--------------------+-----------+---------+---------+------------------+------------------+--------+-------------------+--------+-----+------------+-----------+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+------------+
|         player_name|player_code|team_name|team_code|opponent_team_name|opponent_team_code|position|       kickoff_time|gameweek|round|fixture_code|season_code|goals_scored|goals_conceded|assists|expected_assists|expected_goal_involvements|expected_goals|expected_goals_conceded|penalties_missed|penalties_saved|saves|bps|minutes|yellow_cards|red_cards|own_goals|starts|was_home|influence|creativity|threat|ict_index|clean_sheets|total_points|
+--------------------+-----------+---------+---------+------------------+------------------+--------+---------

### 4.1.5 Union Historical and Current Player Data
In this section we union historical and current player data.

In [16]:
# Union historical and current player data
player_data = cleaned_historical_player_data.union(cleaned_current_player_data)

# Show the combined DataFrame
player_data.show()

+--------------------+-----------+-------------+---------+------------------+------------------+--------+-------------------+--------+-----+------------+-----------+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+------------+
|         player_name|player_code|    team_name|team_code|opponent_team_name|opponent_team_code|position|       kickoff_time|gameweek|round|fixture_code|season_code|goals_scored|goals_conceded|assists|expected_assists|expected_goal_involvements|expected_goals|expected_goals_conceded|penalties_missed|penalties_saved|saves|bps|minutes|yellow_cards|red_cards|own_goals|starts|was_home|influence|creativity|threat|ict_index|clean_sheets|total_points|
+--------------------+-----------+-------------+---------+------------------+------------------+------

### 4.2 Transform & Clean Team Data
In this section we'll transform and clean team data.

**Team Dimensions**
* Team name
* Team code
* Opponent team name
* Opponent team code
* Kickoff time
* Gameweek
* Season code

**Team Metrics**
* Goals scored
* Goals conceded
* Assists
* Expected assists
* Expected goal involvements
* Expected goals
* Expected goals conceded
* Penalties missed
* Penalties saved
* Saves
* BPS (Bonus Points System)
* Points
* Yellow cards
* Red cards
* Own goals
* Clean sheets

### 4.2.1 Aggregate Player Data To Team Data
In this section we aggregate player data to get the data for each team.

In [17]:
# Unique keys for team data
team_data_keys = [
    "fixture_code",
    "team_code",
    "opponent_team_code",
    "kickoff_time",
    "season_code"
]

# Calculate the team data
team_data = player_data\
    .groupBy(team_data_keys)\
    .agg(
        sum("goals_scored").alias("team_goals_scored"),
        max("goals_conceded").alias("team_goals_conceded"),
        sum("assists").alias("team_assists"),
        sum("expected_assists").alias("team_expected_assists"),
        sum("expected_goal_involvements").alias("team_expected_goal_involvements"),
        sum("expected_goals").alias("team_expected_goals"),
        max("expected_goals_conceded").alias("team_expected_goals_conceded"),
        sum("penalties_missed").alias("team_penalties_missed"),
        sum("penalties_saved").alias("team_penalties_saved"),
        sum("saves").alias("team_saves"),
        sum("bps").alias("team_bps"),
        sum("total_points").alias("team_total_points"),
        sum("yellow_cards").alias("team_yellow_cards"),
        sum("red_cards").alias("team_red_cards"),
        sum("own_goals").alias("team_own_goals"),
        max("clean_sheets").alias("team_clean_sheets"),
    )

# Print number of rows
print(f"Number of rows in player_data: {player_data.count()}")
print(f"Number of rows in team_data: {team_data.count()}")

# Show the team dataframe
team_data.show()


Number of rows in player_data: 106302
Number of rows in team_data: 2098
+------------+---------+------------------+-------------------+-----------+-----------------+-------------------+------------+---------------------+-------------------------------+-------------------+----------------------------+---------------------+--------------------+----------+--------+-----------------+-----------------+--------------+--------------+-----------------+
|fixture_code|team_code|opponent_team_code|       kickoff_time|season_code|team_goals_scored|team_goals_conceded|team_assists|team_expected_assists|team_expected_goal_involvements|team_expected_goals|team_expected_goals_conceded|team_penalties_missed|team_penalties_saved|team_saves|team_bps|team_total_points|team_yellow_cards|team_red_cards|team_own_goals|team_clean_sheets|
+------------+---------+------------------+-------------------+-----------+-----------------+-------------------+------------+---------------------+--------------------------

### 4.2.2 Clean Team Data
In this section we clean team data.

In [18]:
team_data_columns = [
    "team_code",
    "opponent_team_code",
    "fixture_code",
    "kickoff_time",
    "season_code",
    "team_goals_scored",
    "team_goals_conceded",
    "team_assists",
    "team_expected_assists",
    "team_expected_goal_involvements",
    "team_expected_goals",
    "team_expected_goals_conceded",
    "team_penalties_missed",
    "team_penalties_saved",
    "team_saves",
    "team_bps",
    "team_yellow_cards",
    "team_red_cards",
    "team_own_goals",
    "team_clean_sheets",
    "team_total_points",
]

# Select required columns
team_data = team_data.select(team_data_columns)

# Show the selected columns
team_data.show()

+---------+------------------+------------+-------------------+-----------+-----------------+-------------------+------------+---------------------+-------------------------------+-------------------+----------------------------+---------------------+--------------------+----------+--------+-----------------+--------------+--------------+-----------------+-----------------+
|team_code|opponent_team_code|fixture_code|       kickoff_time|season_code|team_goals_scored|team_goals_conceded|team_assists|team_expected_assists|team_expected_goal_involvements|team_expected_goals|team_expected_goals_conceded|team_penalties_missed|team_penalties_saved|team_saves|team_bps|team_yellow_cards|team_red_cards|team_own_goals|team_clean_sheets|team_total_points|
+---------+------------------+------------+-------------------+-----------+-----------------+-------------------+------------+---------------------+-------------------------------+-------------------+----------------------------+-----------------

## 5. Feature Engineering
We'll engineer rolling window features to capture recent and longer-term performance trends. For example, we'll calculate rolling averages for player statistics like total points, goals scored, etc.

**Features**
* [team/player] Goals scored [last 4/last 5-16]
* [team/player] Goals conceded [last 4/last 5-16]
* [team/player] Assists [last 4/last 5-16]
* [team/player] Expected assists [last 4/last 5-16]
* [team/player] Expected goal involvements [last 4/last 5-16]
* [team/player] Expected goals [last 4/last 5-16]
* [team/player] Expected goals conceded [last 4/last 5-16]
* [team/player] Penalties scored [last 4/last 5-16]
* [team/player] Penalties missed [last 4/last 5-16]
* [team/player] Penalties saved [last 4/last 5-16]
* [team/player] Saves [last 4/last 5-16]
* [team/player] BPS (Bonus Points System) [last 4/last 5-16]
* [team/player] Points [last 4/last 5-16]
* [team/player] Minutes [last 4/last 5-16]
* [team/player] Yellow cards [last 4/last 5-16]
* [team/player] Red cards [last 4/last 5-16]
* [team/player] Own goals [last 4/last 5-16]
* [team/player] Starts [last 4/last 5-16]
* [team/player] Was home [last 4/last 5-16]
* [team/player] Influence [last 4/last 5-16]
* [team/player] Creativity [last 4/last 5-16]
* [team/player] Threat [last 4/last 5-16]
* [team/player] ICT index [last 4/last 5-16]
* [team/player] Clean sheets [last 4/last 5-16]

### 5.1 Build Player Features
In this section we build player features using a rolling window.

In [19]:
# Define a window specification for recent performance (e.g., last 4 weeks)
# We define a window to calculate rolling metrics over the last 4 weeks.
recent_window = Window.partitionBy("player_code").orderBy("kickoff_time").rowsBetween(Window.currentRow - 4, Window.currentRow - 1)

# Define a window specification for longer-term performance (e.g., last 4 to 12 weeks)
# We define a window to calculate rolling metrics over the last 4 to 12 weeks.
long_term_window = Window.partitionBy("player_code").orderBy("kickoff_time").rowsBetween(Window.currentRow - 12, Window.currentRow - 5)

# List of metrics to get rolling metrics for
metrics = [
    "goals_scored",
    "goals_conceded",
    "assists",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals",
    "expected_goals_conceded",
    "penalties_missed",
    "penalties_saved",
    "saves",
    "bps",
    "minutes",
    "yellow_cards",
    "red_cards",
    "own_goals",
    "starts",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "clean_sheets",
    "total_points",
]

# Make a copy of player_data to calculate features
player_features = player_data

# Calculate rolling metrics with a check for sufficient rows
for metric in metrics:
    # Calculate recent rolling average with row count check
    player_features = player_features.withColumn(
        f"recent_{metric}",
        when(
            count(lit(1)).over(recent_window) >= 4,  # Check if we have at least 4 rows
            sum(metric).over(recent_window)
        ).otherwise(None)  # Return None if not enough rows
    )
    
    # Calculate long-term rolling average with row count check
    player_features = player_features.withColumn(
        f"long_term_{metric}",
        when(
            count(lit(1)).over(long_term_window) >= 8,  # Check for at least 8 rows
            sum(metric).over(long_term_window)
        ).otherwise(None)  # Return None if not enough rows
    )

# Show features dataframe
player_features.show()

+------------+-----------+---------+---------+------------------+------------------+--------+-------------------+--------+-----+------------+-----------+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+------------+-------------------+----------------------+---------------------+------------------------+--------------+-----------------+-----------------------+--------------------------+---------------------------------+------------------------------------+---------------------+------------------------+------------------------------+---------------------------------+-----------------------+--------------------------+----------------------+-------------------------+------------+---------------+----------+-------------+--------------+-----------------+-------------------+

### 5.2 Build Team Features
In this section we build team features using a rolling window.

In [20]:
# Define a window specification for recent performance (e.g., last 4 weeks)
# We define a window to calculate rolling metrics over the last 4 weeks.
recent_window = Window.partitionBy("team_code").orderBy("kickoff_time").rowsBetween(Window.currentRow - 4, Window.currentRow - 1)

# Define a window specification for longer-term performance (e.g., last 4 to 12 weeks)
# We define a window to calculate rolling metrics over the last 4 to 12 weeks.
long_term_window = Window.partitionBy("team_code").orderBy("kickoff_time").rowsBetween(Window.currentRow - 12, Window.currentRow - 5)

# List of metrics to get rolling metrics for
metrics = [
    "team_goals_scored",
    "team_goals_conceded",
    "team_assists",
    "team_expected_assists",
    "team_expected_goal_involvements",
    "team_expected_goals",
    "team_expected_goals_conceded",
    "team_penalties_missed",
    "team_penalties_saved",
    "team_saves",
    "team_bps",
    "team_yellow_cards",
    "team_red_cards",
    "team_own_goals",
    "team_clean_sheets",
    "team_total_points",
]

# Make a copy of team_data to calculate features
team_features = team_data

# Calculate rolling metrics with a check for sufficient rows
for metric in metrics:
    # Calculate recent rolling average with row count check
    team_features = team_features.withColumn(
        f"recent_{metric}",
        when(
            count(lit(1)).over(recent_window) >= 4,  # Check if we have at least 4 rows
            sum(metric).over(recent_window)
        ).otherwise(None)  # Return None if not enough rows
    )
    
    # Calculate long-term rolling average with row count check
    team_features = team_features.withColumn(
        f"long_term_{metric}",
        when(
            count(lit(1)).over(long_term_window) >= 8,  # Check for at least 8 rows
            sum(metric).over(long_term_window)
        ).otherwise(None)  # Return None if not enough rows
    )

# Show features dataframe
team_features.show()

+---------+------------------+------------+-------------------+-----------+-----------------+-------------------+------------+---------------------+-------------------------------+-------------------+----------------------------+---------------------+--------------------+----------+--------+-----------------+--------------+--------------+-----------------+-----------------+------------------------+---------------------------+--------------------------+-----------------------------+-------------------+----------------------+----------------------------+-------------------------------+--------------------------------------+-----------------------------------------+--------------------------+-----------------------------+-----------------------------------+--------------------------------------+----------------------------+-------------------------------+---------------------------+------------------------------+-----------------+--------------------+---------------+------------------+---

### 5.3 Merge Features
In this section we combine player and team features into one feature table 

In [21]:
features = player_features\
    .join(
        team_features,
        (team_data_keys),
        "inner"
    )

print(f"Number of rows in player_features: {player_features.count()}")
print(f"Number of rows in team_features: {team_features.count()}")
print(f"Number of rows in features: {features.count()}")

# Show the merged dataframe
features.show()

Number of rows in player_features: 106302
Number of rows in team_features: 2098
Number of rows in features: 106302


+------------+---------+------------------+-------------------+-----------+--------------------+-----------+-------------+------------------+--------+--------+-----+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+------------+-------------------+----------------------+---------------------+------------------------+--------------+-----------------+-----------------------+--------------------------+---------------------------------+------------------------------------+---------------------+------------------------+------------------------------+---------------------------------+-----------------------+--------------------------+----------------------+-------------------------+------------+---------------+----------+-------------+--------------+-----------------+--------

### 5.4 Transform Categorical Features
In this section we transform categorical features into numerical features:
- position
- was_home

In [22]:
# One-hot encoding on position
features = features\
    .withColumn("position_GK",
                when(col("position") == "GK", 1).otherwise(0)
    )\
    .withColumn("position_DEF",
                when(col("position") == "DEF", 1).otherwise(0)
    )\
    .withColumn("position_MID",
                when(col("position") == "MID", 1).otherwise(0)
    )\
    .withColumn("position_FWD",
                when(col("position") == "FWD", 1).otherwise(0)
    )

# Show distinct combinations
features.select("position", "position_GK", "position_DEF", "position_MID", "position_FWD").distinct().show()

+--------+-----------+------------+------------+------------+
|position|position_GK|position_DEF|position_MID|position_FWD|
+--------+-----------+------------+------------+------------+
|     FWD|          0|           0|           0|           1|
|      GK|          1|           0|           0|           0|
|     MID|          0|           0|           1|           0|
|     DEF|          0|           1|           0|           0|
|    NULL|          0|           0|           0|           0|
+--------+-----------+------------+------------+------------+



In [23]:
# One-hot encoding on was_home
features = features\
    .withColumn("was_home_true",
                when(col("was_home") == True, 1).otherwise(0)
    )\
    .withColumn("was_home_false",
                when(col("was_home") == False, 1).otherwise(0)
    )

# Show distinct combinations
features.select("was_home", "was_home_true", "was_home_false").distinct().show()

+--------+-------------+--------------+
|was_home|was_home_true|was_home_false|
+--------+-------------+--------------+
|    true|            1|             0|
|   false|            0|             1|
+--------+-------------+--------------+



## 6. Write Data
Finally, we'll write the features to a parquet file for use in the subsequent notebooks.

In [ ]:
# Define output directory
# We define the directory where the processed data will be saved.
data_source = "features"
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"../data/processed/{data_source}/{current_datetime}"
filename = "features.parquet"
filepath = os.path.join(output_dir, filename)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Create the directory if it doesn't exist
# We create the directory if it doesn't exist.
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# # Write the transformed DataFrames to Parquet files
# # We save the processed data as parquet files.
features.write.parquet(f"{filepath}", mode="overwrite")

print(f"Features saved to {output_dir}")

25/03/30 10:52:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/03/30 10:52:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
25/03/30 10:52:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
25/03/30 10:52:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
25/03/30 10:52:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
25/03/30 10:52:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
25/03/30 10:52:07 WARN MemoryManager: Total allocation exceeds 95.

training features saved to ../data/processed/features/20250330_105203


25/03/30 10:52:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
25/03/30 10:52:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
25/03/30 10:52:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
25/03/30 10:52:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
25/03/30 10:52:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
25/03/30 10:52:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


25/03/30 12:39:47 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 994914 ms exceeds timeout 120000 ms
25/03/30 12:39:47 WARN SparkContext: Killing executors is not supported by current scheduler.
25/03/30 14:02:54 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$